# Task 4: Improved eLCS-Based System

### 4.1 Improvement overview

The original eLCS baseline struggled with the severe class imbalance and predicted almost every testing record as class 0. The improved pipeline reduces the class-0 training records to 25,000 while retaining every available class-1 training record. No synthetic records are created, and the testing dataset remains unchanged.

Three eLCS configurations are compared:

1. Original eLCS trained on the full preprocessed and imbalanced training data.
2. Original eLCS trained on the undersampled data using its default parameters.
3. Improved eLCS trained on the undersampled data using a population size of 8,000 and 50,000 learning iterations.

Using the second configuration separately allows the effect of undersampling to be distinguished from the effect of the larger rule population and longer learning process.

### 4.2 Import packages and load the preprocessed datasets

The full and undersampled training datasets produced in Task 3 are loaded. Both versions use the same preprocessing and predictor columns.

The unchanged preprocessed test set is loaded once and used for every experiment. This ensures that differences in performance are caused by the training data or eLCS configuration rather than different testing records.

In [1]:
from pathlib import Path

import sys
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# locates the project root
if Path.cwd().name in {"notebooks", "notebooks-phase2"}:
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"

# loads the full preprocessed training data
X_train_full = pd.read_csv(
    DATA_DIR / "X_train_preprocessed_full.csv"
)
y_train_full = pd.read_csv(
    DATA_DIR / "y_train_preprocessed_full.csv"
).squeeze()

# loads the undersampled preprocessed training data
X_train_undersampled = pd.read_csv(
    DATA_DIR / "X_train_preprocessed_undersampled.csv"
)
y_train_undersampled = pd.read_csv(
    DATA_DIR / "y_train_preprocessed_undersampled.csv"
).squeeze()

# loads the single unchanged preprocessed test set
X_test = pd.read_csv(
    DATA_DIR / "X_test_preprocessed.csv"
)
y_test = pd.read_csv(
    DATA_DIR / "y_test_preprocessed.csv"
).squeeze()

print("Full training shape:", X_train_full.shape)
print("Undersampled training shape:", X_train_undersampled.shape)
print("Testing shape:", X_test.shape)

print("\nFull training distribution:")
print(y_train_full.value_counts().sort_index())

print("\nUndersampled training distribution:")
print(y_train_undersampled.value_counts().sort_index())

print("\nTesting distribution:")
print(y_test.value_counts().sort_index())

Full training shape: (222960, 34)
Undersampled training shape: (29713, 34)
Testing shape: (55741, 34)

Full training distribution:
heart_disease_target
0    218247
1      4713
Name: count, dtype: int64

Undersampled training distribution:
heart_disease_target
0    25000
1     4713
Name: count, dtype: int64

Testing distribution:
heart_disease_target
0    54563
1     1178
Name: count, dtype: int64


### 4.3 Validate the Task 3 datasets

Before model training, the loaded datasets are checked for matching columns, numeric values, missing values, and the expected class distributions.

The undersampled training data must contain exactly 25,000 class-0 records and every class-1 training record. The test set must retain its original imbalanced distribution.

In [2]:
# confirms that all datasets contain the same predictors
assert list(X_train_full.columns) == list(
    X_train_undersampled.columns
)
assert list(X_train_full.columns) == list(X_test.columns)

# confirms that predictors and targets have matching row counts
assert len(X_train_full) == len(y_train_full)
assert len(X_train_undersampled) == len(
    y_train_undersampled
)
assert len(X_test) == len(y_test)

# confirms that there are no missing values are present
assert X_train_full.isna().sum().sum() == 0
assert X_train_undersampled.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

assert y_train_full.isna().sum() == 0
assert y_train_undersampled.isna().sum() == 0
assert y_test.isna().sum() == 0

# confirms the expected undersampled distribution
assert int((y_train_undersampled == 0).sum()) == 25_000
assert int((y_train_undersampled == 1).sum()) == int(
    (y_train_full == 1).sum()
)

print("All Task 3 dataset checks passed.")
print("The same test set will be used for all three models.")

All Task 3 dataset checks passed.
The same test set will be used for all three models.


### 4.4 Import the supplied eLCS implementation

The original supplied scikit-eLCS version 1.2.4 is imported from the project directory. The underlying eLCS source code is not modified.

In [3]:
# locates the supplied eLCS implementation
ELCS_ROOT = PROJECT_ROOT / "external" / "scikit-eLCS-master"

if not ELCS_ROOT.exists():
    raise FileNotFoundError(
        f"Expected the supplied eLCS package at: {ELCS_ROOT}"
    )

if str(ELCS_ROOT) not in sys.path:
    sys.path.insert(0, str(ELCS_ROOT))

from skeLCS import eLCS

print("eLCS imported from:", ELCS_ROOT)

eLCS imported from: /Users/fatehbhular/Multi_Disease_Prediction_ENGE707/external/scikit-eLCS-master


### 4.5 Confirm the three experimental configurations

The first two configurations use the original eLCS default parameters. Their only difference is whether the full or undersampled training data is used.

The final improved configuration uses the undersampled training data together with a maximum rule-population size of 8,000 and 50,000 learning iterations. All other eLCS parameters remain at their original default values.

In [4]:
# creates example models to confirm their parameters
default_model = eLCS(random_state=3)

improved_model = eLCS(
    N=8000,
    learning_iterations=50000,
    random_state=3,
)

parameter_comparison = pd.DataFrame(
    {
        "Configuration": [
            "Default eLCS",
            "Improved eLCS",
        ],
        "Training data": [
            "Full or undersampled",
            "Undersampled",
        ],
        "N": [
            default_model.get_params()["N"],
            improved_model.get_params()["N"],
        ],
        "Learning iterations": [
            default_model.get_params()["learning_iterations"],
            improved_model.get_params()["learning_iterations"],
        ],
    }
)

display(parameter_comparison)

,Configuration,Training data,N,Learning iterations
0,Default eLCS,Full or undersampled,1000,10000
1,Improved eLCS,Undersampled,8000,50000


### 4.6 Prepare the datasets and experimental seeds

The DataFrames are converted to NumPy arrays for eLCS. The same ten seeds used for the original raw-data baseline are reused for all three Task 4 configurations.

The Task 3 train-test split and undersampled training dataset remain fixed. In Task 4, each seed controls the random behaviour of eLCS. Every configuration is evaluated using the same unchanged test set.

In [5]:
# converts the full training data to NumPy arrays
X_train_full_array = X_train_full.to_numpy(dtype=float)
y_train_full_array = y_train_full.to_numpy(dtype=int)

# converts the undersampled training data to NumPy arrays
X_train_undersampled_array = (
    X_train_undersampled.to_numpy(dtype=float)
)
y_train_undersampled_array = (
    y_train_undersampled.to_numpy(dtype=int)
)

# converts the unchanged test data to NumPy arrays
X_test_array = X_test.to_numpy(dtype=float)
y_test_array = y_test.to_numpy(dtype=int)

# uses the same seeds as the original baseline model
SEEDS = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]

print("Full training array:", X_train_full_array.shape)
print(
    "Undersampled training array:",
    X_train_undersampled_array.shape,
)
print("Testing array:", X_test_array.shape)
print("Seeds:", SEEDS)

Full training array: (222960, 34)
Undersampled training array: (29713, 34)
Testing array: (55741, 34)
Seeds: [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]


### 4.7 Original eLCS on the full preprocessed training data

The first configuration trains the original eLCS on the complete preprocessed training data. It uses the original default eLCS parameters.

This experiment measures the effect of preprocessing without introducing undersampling or changing the eLCS configuration.

In [ ]:
# runs the default eLCS on the full preprocessed training data
def run_full_preprocessed(seed):
    model = eLCS(random_state=seed)

    start_time = time.perf_counter()

    model.fit(
        X_train_full_array,
        y_train_full_array,
    )

    training_seconds = (
        time.perf_counter() - start_time
    )

    predictions = np.asarray(
        model.predict(X_test_array)
    ).astype(int)

    probabilities = model.predict_proba(
        X_test_array
    )[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test_array,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "Seed": seed,
        "Accuracy": accuracy_score(
            y_test_array,
            predictions,
        ),
        "Precision": precision_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "F1": f1_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_array,
            predictions,
        ),
        "ROC-AUC": roc_auc_score(
            y_test_array,
            probabilities,
        ),
        "PR-AUC": average_precision_score(
            y_test_array,
            probabilities,
        ),
        "True negatives": tn,
        "False positives": fp,
        "False negatives": fn,
        "True positives": tp,
        "Positive predictions": int(
            (predictions == 1).sum()
        ),
        "Training seconds": training_seconds,
    }

### 4.8 Original eLCS on the undersampled training data

The second configuration uses the original default eLCS parameters but trains on the undersampled dataset.

Comparing this configuration with the full preprocessed configuration isolates the effect of majority-class undersampling. It does not yet include the increased population size or additional learning iterations.

In [ ]:
# runs the default eLCS on the undersampled training data
def run_undersampling_only(seed):
    model = eLCS(random_state=seed)

    start_time = time.perf_counter()

    model.fit(
        X_train_undersampled_array,
        y_train_undersampled_array,
    )

    training_seconds = (
        time.perf_counter() - start_time
    )

    predictions = np.asarray(
        model.predict(X_test_array)
    ).astype(int)

    probabilities = model.predict_proba(
        X_test_array
    )[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test_array,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "Seed": seed,
        "Accuracy": accuracy_score(
            y_test_array,
            predictions,
        ),
        "Precision": precision_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "F1": f1_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_array,
            predictions,
        ),
        "ROC-AUC": roc_auc_score(
            y_test_array,
            probabilities,
        ),
        "PR-AUC": average_precision_score(
            y_test_array,
            probabilities,
        ),
        "True negatives": tn,
        "False positives": fp,
        "False negatives": fn,
        "True positives": tp,
        "Positive predictions": int(
            (predictions == 1).sum()
        ),
        "Training seconds": training_seconds,
    }

### 4.9 Final improved eLCS

The final configuration trains on the same undersampled dataset but increases the eLCS population size from 1,000 to 8,000 and the learning iterations from 10,000 to 50,000.

The larger population allows eLCS to retain a wider set of candidate rules, while the additional learning iterations give those rules more opportunities to be evaluated and evolved. This may improve minority-class recognition and rule quality, although it is also expected to increase training time.

In [ ]:
# runs the improved eLCS on the undersampled training data
def run_improved_elcs(seed):
    model = eLCS(
        N=8000,
        learning_iterations=50000,
        random_state=seed,
    )

    start_time = time.perf_counter()

    model.fit(
        X_train_undersampled_array,
        y_train_undersampled_array,
    )

    training_seconds = (
        time.perf_counter() - start_time
    )

    predictions = np.asarray(
        model.predict(X_test_array)
    ).astype(int)

    probabilities = model.predict_proba(
        X_test_array
    )[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test_array,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "Seed": seed,
        "Accuracy": accuracy_score(
            y_test_array,
            predictions,
        ),
        "Precision": precision_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "F1": f1_score(
            y_test_array,
            predictions,
            zero_division=0,
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_array,
            predictions,
        ),
        "ROC-AUC": roc_auc_score(
            y_test_array,
            probabilities,
        ),
        "PR-AUC": average_precision_score(
            y_test_array,
            probabilities,
        ),
        "True negatives": tn,
        "False positives": fp,
        "False negatives": fn,
        "True positives": tp,
        "Positive predictions": int(
            (predictions == 1).sum()
        ),
        "Training seconds": training_seconds,
    }

### 4.10 Run the repeated experiments

Each configuration is run once for each of the ten recorded seeds. The result of every run is retained so that average performance, variation between runs, and statistical significance can be evaluated later.

These experiments may take considerable time, particularly the final improved configuration with 50,000 learning iterations.

In [ ]:
# stores the results for each configuration
full_preprocessed_runs = []
undersampling_only_runs = []
improved_elcs_runs = []

# runs the default eLCS on the full preprocessed data
for seed in SEEDS:
    print(f"Full preprocessed data: starting seed {seed}")

    result = run_full_preprocessed(seed)
    full_preprocessed_runs.append(result)

    print(
        f"Completed seed {seed}: "
        f"balanced accuracy="
        f"{result['Balanced accuracy']:.4f}, "
        f"positive predictions="
        f"{result['Positive predictions']}"
    )

full_preprocessed_df = pd.DataFrame(
    full_preprocessed_runs
)

display(full_preprocessed_df)

In [ ]:
# runs the default eLCS on the undersampled data
for seed in SEEDS:
    print(f"Undersampling only: starting seed {seed}")

    result = run_undersampling_only(seed)
    undersampling_only_runs.append(result)

    print(
        f"Completed seed {seed}: "
        f"balanced accuracy="
        f"{result['Balanced accuracy']:.4f}, "
        f"positive predictions="
        f"{result['Positive predictions']}"
    )

undersampling_only_df = pd.DataFrame(
    undersampling_only_runs
)

display(undersampling_only_df)

In [ ]:
# runs the improved eLCS on the undersampled data
for seed in SEEDS:
    print(f"Improved eLCS: starting seed {seed}")

    result = run_improved_elcs(seed)
    improved_elcs_runs.append(result)

    print(
        f"Completed seed {seed}: "
        f"balanced accuracy="
        f"{result['Balanced accuracy']:.4f}, "
        f"positive predictions="
        f"{result['Positive predictions']}"
    )

improved_elcs_df = pd.DataFrame(
    improved_elcs_runs
)

display(improved_elcs_df)

### 4.11 Validate the experimental results

The result tables are checked to confirm that all ten seeds were completed, every confusion matrix accounts for the entire test set, and no required metric is missing.

In [ ]:
result_tables = {
    "Full preprocessed": full_preprocessed_df,
    "Undersampling only": undersampling_only_df,
    "Improved eLCS": improved_elcs_df,
}

required_metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced accuracy",
    "ROC-AUC",
    "PR-AUC",
    "Training seconds",
]

for model_name, results in result_tables.items():
    assert len(results) == len(SEEDS)
    assert results["Seed"].nunique() == len(SEEDS)
    assert not results[required_metrics].isna().any().any()

    confusion_totals = (
        results["True negatives"]
        + results["False positives"]
        + results["False negatives"]
        + results["True positives"]
    )

    assert (confusion_totals == len(y_test)).all()

    print(f"{model_name}: all result checks passed.")

### 4.12 Summarise and compare the results

The mean, standard deviation, minimum, and maximum are calculated across the ten runs for each configuration.

For this imbalanced classification problem, balanced accuracy, recall, F1, ROC-AUC, and PR-AUC are more informative than accuracy alone. Training time is also reported because increasing the population size and learning iterations introduces additional computational cost.

In [ ]:
summary_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced accuracy",
    "ROC-AUC",
    "PR-AUC",
    "Training seconds",
]

# summarises the full preprocessed default-eLCS results
full_preprocessed_summary = (
    full_preprocessed_df[summary_columns]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

full_preprocessed_summary.insert(
    0,
    "Configuration",
    "Full preprocessed - default eLCS",
)

# summarises the undersampling-only results
undersampling_only_summary = (
    undersampling_only_df[summary_columns]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

undersampling_only_summary.insert(
    0,
    "Configuration",
    "Undersampling only - default eLCS",
)

# summarises the final improved-eLCS results
improved_elcs_summary = (
    improved_elcs_df[summary_columns]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

improved_elcs_summary.insert(
    0,
    "Configuration",
    "Undersampled - improved eLCS",
)

# combines all of the summaries into one comparison table
combined_summary = pd.concat(
    [
        full_preprocessed_summary,
        undersampling_only_summary,
        improved_elcs_summary,
    ],
    ignore_index=True,
)

display(combined_summary)

### 4.13 Save the Task 4 results

The individual ten-run result tables and the combined summary are saved for later statistical testing, model comparison, report tables, and result interpretation.

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "improved_elcs_undersampling"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# saves the ten-run result tables
full_preprocessed_df.to_csv(
    OUTPUT_DIR / "full_preprocessed_default_runs.csv",
    index=False,
)

undersampling_only_df.to_csv(
    OUTPUT_DIR / "undersampling_only_default_runs.csv",
    index=False,
)

improved_elcs_df.to_csv(
    OUTPUT_DIR / "undersampled_improved_elcs_runs.csv",
    index=False,
)

# saves the combined descriptive summary
combined_summary.to_csv(
    OUTPUT_DIR / "task4_comparison_summary.csv",
    index=False,
)

print("Saved Task 4 results to:", OUTPUT_DIR.resolve())